# Activity 4 — Google Trends

Google Trends tells you how much people in a country searched for a term, month by
month, since 2004. It is free and it has **no publication lag**: September's searches
are available in the first days of October. That is exactly what a nowcast needs.

In this notebook you will:

1. download a few search terms for Trinidad and Tobago with `trendspy`;
2. compare them with the full set we ship in `data/gtrends.csv`;
3. turn search interest into a quarterly predictor and set it against GDP.

**Kernel: `nwcst`** (it has `trendspy`).

---

### Three things to know before you download anything

**1. `pytrends` is dead.** For years everyone used `pytrends`. It was archived on
2025-04-17 and is no longer maintained: it now returns `429 Too Many Requests` on the
*first* call, because its session handling is out of date — not because you were
blocked. If an old notebook imports `pytrends`, that is why it fails. We use
`trendspy`, a maintained replacement (no API key).

**2. One term per query, never batches.** Google rescales every query to 0–100 *within
that query*. Ask for five terms at once and each is scaled to the most-searched of the
five: the small ones collapse to near-constant integers (a max of 1 or 2) and carry no
signal. Asked alone, each term spans the full 0–100 range. Each term becomes its own
standardised predictor anyway, so nothing is lost.

**3. Google rate-limits.** `trendspy` is unofficial. At one second between queries,
the ~40th call returns `429`; after that Google keeps blocking for tens of minutes.
**In this room everyone shares one internet address**, so we will hit that limit much
sooner than one laptop would. The rule for today:

> **Download only 2–3 terms each**, with at least 2 seconds between queries.
> The full set of 25 terms is already in `data/gtrends.csv`. If Google blocks you,
> carry on with that file, or use the manual download in Part B.

**Updating means re-downloading.** Each download is rescaled 0–100 over its own
window, so you cannot append a new month to an old series: the whole term is replaced.

In [ ]:
# ===========================================================================
# CONFIG LOADER  --  edit config.py, NOT this cell.
# Gives you `cfg` (every setting) and `P` (the standard folders).
# ===========================================================================
import os, sys, importlib, importlib.util

# config.py lives in activity/. From activity/ that is the current folder;
# from sols/ it is next door.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))
_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)      # public/raw, public/data, public/output, ...

PATH_RAW  = P["raw"]      # your downloads are written here
PATH_DATA = P["data"]     # keyword list and the shipped gtrends.csv
print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE})")
print(f"         raw  : {PATH_RAW}")
print(f"         data : {PATH_DATA}")

In [ ]:
# trendspy is imported in the download cell, so Part B works without it.
import re, time, random, unicodedata, warnings
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ---- parameters ------------------------------------------------------------
GT_START     = cfg.GT_START                         # 2004-01-01 = "all" on the website
GT_END       = date.today().strftime("%Y-%m-%d")
GT_TIMEFRAME = f"{GT_START} {GT_END}"
GT_PAUSE     = 5.0       # seconds between queries. Never below 2.

GT_CSV    = os.path.join(PATH_RAW, "gtrends.csv")        # what YOU download
GT_SHIP   = os.path.join(PATH_DATA, "gtrends.csv")       # the full shipped set
GT_MANUAL = os.path.join(PATH_RAW, "gtrends_manual")

# Google's current month is partial: it is dropped from every save.
THIS_MONTH = pd.Timestamp(date.today()).to_period("M").to_timestamp()


def slug(s):
    """'First Citizens Bank' -> 'first_citizens_bank'."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


def save(series):
    """Write a dict column -> Series to raw/gtrends.csv (zeros stored as empty)."""
    if not series:
        return None
    df = pd.DataFrame(series)
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    df.index.name = "date"
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df[df.index < THIS_MONTH]
    # Trinidad's convention: a zero month is "too few searches to report", not zero.
    df = df.mask(df == 0)
    df.to_csv(GT_CSV)
    return df


print(f"period = {GT_TIMEFRAME}")

## Step 1: the keyword list

`data/TT_GTrends_Keywords.csv` has one row per term: the `keyword` as typed in Google,
the `geo` (country code; empty = worldwide) and the `column` name it gets in the data.
Look at the list: shopping (`amazon`, `shein`, `pricesmart`), jobs (`caribbean jobs`,
`linkedin`), banks, restaurants, tourism (`tobago hotel`, `flights trinidad` searched
**worldwide** — that is foreign demand for the country).

In [ ]:
kw = pd.read_csv(os.path.join(PATH_DATA, cfg.GT_KEYWORDS), dtype=str).fillna("")
kw["keyword"] = kw["keyword"].str.strip()
kw = kw[kw["keyword"] != ""].drop_duplicates(subset=["keyword", "geo"])
kw

### Exercise 1 — pick your terms

Choose **two or three** terms. Take some from the list above, and add **one term of
your own** that you think tracks economic activity in Trinidad and Tobago (a retailer,
a job site, a holiday, a product...). For a new term, give `geo = "TT"` and a column
name `gtrends_<term>_tt`.

In [ ]:
# Edit this list: (keyword, geo, column). Keep it to 2-3 terms.
MY_TERMS = [
    ("amazon",         "TT", "gtrends_amazon_tt"),
    ("caribbean jobs", "TT", "gtrends_caribbean_jobs_tt"),
    # ("your term here", "TT", "gtrends_your_term_here_tt"),
]
print(len(MY_TERMS), "term(s) to download")

## Step 2: download with `trendspy`

One query per term, a pause between queries, and the file is **saved after every
term** — so if Google blocks you halfway, nothing already downloaded is lost.
If you get a `429`, stop: wait, or go to Part B.

In [ ]:
from trendspy import Trends              # if missing:  pip install trendspy

assert len(MY_TERMS) <= 5, "Keep it to a handful of terms -- we share one IP address."

tr = Trends(request_delay=max(GT_PAUSE, 2.0))
series = ({c: s for c, s in pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"]).items()}
          if os.path.exists(GT_CSV) else {})
failed = []

for i, (term, geo, col) in enumerate(MY_TERMS, 1):
    tr.request_delay = random.uniform(GT_PAUSE, GT_PAUSE + 1.0)
    try:
        d = tr.interest_over_time([term], geo=geo, timeframe=GT_TIMEFRAME)
        c = [c for c in d.columns if c != "isPartial"][0]
        s = pd.to_numeric(d[c], errors="coerce")
        s.index = pd.to_datetime(d.index).tz_localize(None).normalize()
        series[col] = s
        save(series)                                # <- saved after every term
        print(f"  [{i}/{len(MY_TERMS)}] {term:20} ok  ({(s > 0).mean():.0%} non-zero months)")
    except Exception as e:
        failed.append(term)
        print(f"  [{i}/{len(MY_TERMS)}] {term:20} !! {type(e).__name__}: {str(e)[:80]}")
        if "429" in str(e) or "Too Many" in str(e):
            print("      Google rate limit. Use data/gtrends.csv, or Part B below.")
            break

mine = save(series)
if mine is not None:
    print(f"\nraw/gtrends.csv: {mine.shape[1]} series, "
          f"{mine.index.min().date()} to {mine.index.max().date()}")

## Step 3: what you downloaded vs. what we ship

`data/gtrends.csv` holds all 25 terms, downloaded one by one for the research
pipeline. If a term is in both, the two curves should have the same *shape* — but not
necessarily the same numbers: Google samples its data, and every download is
rescaled to its own maximum.

In [ ]:
ship = pd.read_csv(GT_SHIP, index_col="date", parse_dates=["date"]).sort_index()
print(f"shipped: {ship.shape[1]} series, {ship.index.min().date()} to {ship.index.max().date()}")

mine = (pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"])
        if os.path.exists(GT_CSV) else pd.DataFrame())
common = [c for c in mine.columns if c in ship.columns]

if common:
    fig, axes = plt.subplots(len(common), 1, figsize=(11, 2.6 * len(common)), squeeze=False)
    for ax, c in zip(axes[:, 0], common):
        ship[c].plot(ax=ax, lw=1.2, label="shipped")
        mine[c].plot(ax=ax, lw=1.0, ls="--", label="your download")
        ax.set_title(c, fontsize=10); ax.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("Nothing downloaded in common with the shipped file -- plotting the shipped set.")

show = [c for c in ["gtrends_amazon_tt", "gtrends_kfc_tt", "gtrends_caribbean_jobs_tt",
                    "gtrends_tobago_tt", "gtrends_rbc_tt"] if c in ship.columns]
ax = ship[show].rolling(12, min_periods=6).mean().plot(figsize=(11, 4), lw=1.4)
ax.set_title(f"Google Trends, {cfg.COUNTRY} -- 12-month mean of search interest")
ax.set_ylabel("0-100"); ax.legend(fontsize=8, ncol=3, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## Step 4: from searches to a GDP predictor

GDP is quarterly; Trends is monthly. To set them side by side:

1. keep terms with enough data (at least 75% of months non-empty since 2012);
2. average each term over the quarter;
3. take **year-on-year** growth (Trends has strong seasonality — Christmas, Carnival —
   and year-on-year growth removes most of it);
4. average the terms into one composite index and correlate it with GDP growth.

GDP is in `data/gdp.csv` (real GDP, quarterly; the date is the first day of the
quarter's **last** month — 2026Q1 is `2026-03-01`).

### Exercise 2 — build the composite and correlate it with GDP

Fill in the blanks. Hint: `df.resample("QE").mean()` gives quarterly means;
`pct_change(4)` gives year-on-year growth on quarterly data.

In [ ]:
gdp = pd.read_csv(os.path.join(PATH_DATA, "gdp.csv"), parse_dates=["date"])
gdp["q"] = gdp["date"].dt.to_period("Q")
gdp = gdp.set_index("q")["rgdp"]

gt = ship.loc["2012":]
keep = gt.columns[gt.notna().mean() >= 0.75]
print(f"{len(keep)} of {gt.shape[1]} terms kept")

gt_q = ...            # 1. quarterly mean of gt[keep]
gt_q.index = gt_q.index.to_period("Q")
gt_yoy = ...          # 2. year-on-year growth, in %
composite = ...       # 3. mean across terms

both = pd.concat([composite.rename("gtrends"), (gdp.pct_change(4) * 100).rename("gdp")],
                 axis=1).dropna()
print(both.corr())

**Questions to discuss**

* How much of the correlation comes from 2020–21? A single shock that moves everything
  makes any series look like a good predictor.
* With ~50 quarters and 25 terms, how many terms would correlate with GDP at 0.4
  *by chance*? This is why the nowcast does variable selection (Activity 10) instead
  of picking the best-looking series by eye.

---

# Part B: manual download

When Google keeps returning `429`, download by hand. The next cell prints, for each of
your terms, the Google Trends page with the same period and country. Open it, click the
download arrow (top right of the chart), and save the CSV **without renaming it** in
`raw/gtrends_manual/`. Then run the last cell.

In [ ]:
from urllib.parse import quote

os.makedirs(GT_MANUAL, exist_ok=True)

def trends_link(term, geo):
    period = "all" if GT_START == "2004-01-01" else f"{GT_START}%20{GT_END}"
    return (f"https://trends.google.com/trends/explore?date={period}"
            + (f"&geo={geo}" if geo else "") + f"&q={quote(term)}")

print(f"Save the CSVs in:\n  {GT_MANUAL}\n")
for term, geo, col in MY_TERMS:
    print(f"  {term}  ({geo or 'worldwide'})")
    print(f"    {trends_link(term, geo)}")

## Merge the manual downloads into `raw/gtrends.csv`

Google has used two export formats; both are read:

| file | format |
|---|---|
| `multiTimeline*.csv` | old page: a title line, then `Month`, `term: (Country)` |
| `time_series_<GEO>_*.csv` | current page: `Time` and the term |

Each column takes the `column` name of its term in `MY_TERMS` (or in the keyword
list), so a manual series lands exactly where a downloaded one would.

In [ ]:
import glob

lookup = {slug(t): c for t, g, c in MY_TERMS}
lookup.update({slug(k): c for k, c in zip(kw["keyword"], kw["column"]) if slug(k) not in lookup})

def target_column(term, geo):
    return lookup.get(slug(term), "gtrends_" + slug(term) + "_" + (geo.lower() or "worldwide"))

pieces = []
for f in sorted(glob.glob(os.path.join(GT_MANUAL, "*ultiTimeline*.csv"))):
    d = pd.read_csv(f, skiprows=1).rename(columns={"Month": "date"})
    for c in d.columns[1:]:
        m = re.match(r"(.*?):\s*\((.*)\)\s*$", str(c))
        term, place = (m.group(1), m.group(2)) if m else (str(c), "")
        geo = "" if place.lower() == "worldwide" else cfg.CODE
        pieces.append(d.set_index("date")[c].rename(target_column(term, geo)))
for f in sorted(glob.glob(os.path.join(GT_MANUAL, "time_series_*.csv"))):
    m = re.match(r"time_series_([A-Za-z]+)_", os.path.basename(f))
    geo = m.group(1).upper() if m else ""
    geo = "" if geo in ("GLB", "WORLDWIDE", "WW") else geo
    d = pd.read_csv(f).rename(columns={"Time": "date"})
    for c in d.columns[1:]:
        pieces.append(d.set_index("date")[c].rename(target_column(c, geo)))

if not pieces:
    print("No manual files in raw/gtrends_manual/.")
else:
    manual = pd.concat(pieces, axis=1)
    manual.index = pd.to_datetime(manual.index, errors="coerce")
    manual = manual[manual.index.notna()]
    manual = manual.loc[:, ~manual.columns.duplicated(keep="last")]
    # Google writes "<1" for minimal interest: that is ~0, not missing
    manual = manual.replace("<1", 0).apply(pd.to_numeric, errors="coerce")
    series = ({c: s for c, s in pd.read_csv(GT_CSV, index_col="date", parse_dates=["date"]).items()}
              if os.path.exists(GT_CSV) else {})
    series.update({c: manual[c] for c in manual.columns})
    out = save(series)
    print(f"-> {GT_CSV}: {out.shape[1]} series, {out.index.min().date()} to {out.index.max().date()}")
    print("merged:", list(manual.columns))